In [ ]:
# =============================================================================
# CELL 1 — CONFIGURATION & DATA FETCHING (REFACTORED)
# Quote scan  → Finnhub /quote  (free tier ✓)
# OHLC data   → yfinance        (free, replaces /stock/candle which is 403)
# Outputs: ohlc_data, watchlist_ohlc_data, watchlist_performance
# 
# IMPROVEMENTS:
#   • Session reuse + HTTPAdapter for connection pooling
#   • Exponential backoff retry logic (3 retries, 0.5s base, 2x multiplier)
#   • Longer delays between requests (2+ seconds)
#   • User-Agent headers (mimics real browser)
#   • Increased scan limit (50 tickers → more valid movers)
# =============================================================================

import os
import time
import requests
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from dotenv import load_dotenv
from pytickersymbols import PyTickerSymbols
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Configuration
load_dotenv()

FINNHUB_API_KEY  = os.getenv("FINNHUB_API_KEY", "")
FINNHUB_BASE_URL = "https://finnhub.io/api/v1"
REQUEST_DELAY    = 2.0   # seconds between Finnhub requests (increased from 0.7)
MAX_REQUESTS     = 60    # budget for Finnhub /quote calls (increased from 40)
OHLC_DAYS        = 7     # look-back window for OHLC data
SCAN_LIMIT       = 50    # how many S&P 500 tickers to scan for movers (increased from 20)

SYMBOL_NAMES = {
    "AAPL":  "Apple Inc.",
    "MSFT":  "Microsoft Corporation",
    "NVDA":  "NVIDIA Corporation",
    "TSLA":  "Tesla Inc.",
    "AMZN":  "Amazon Inc.",
    "GOOGL": "Alphabet Inc.",
    "META":  "Meta Platforms",
    "BRK.B": "Berkshire Hathaway",
    "JNJ":   "Johnson & Johnson",
    "V":     "Visa Inc.",
}

WATCHLIST = ["AAPL", "NVDA", "MSFT", "META", "GOOGL"]

if not FINNHUB_API_KEY:
    raise ValueError(
        "FINNHUB_API_KEY not set.\n"
        "1. Visit https://finnhub.io to get a free key.\n"
        "2. Add FINNHUB_API_KEY=<your_key> to a .env file.\n"
        "3. Restart the kernel and re-run."
    )

# Session setup with retry logic
_request_count = 0
_session = None

def _get_session():
    """Create a requests session with connection pooling and exponential backoff."""
    global _session
    if _session is None:
        _session = requests.Session()
        
        # Exponential backoff: 3 retries, 0.5s base, 2x multiplier
        retry = Retry(
            total=3,
            backoff_factor=0.5,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"]
        )
        adapter = HTTPAdapter(max_retries=retry)
        _session.mount("http://", adapter)
        _session.mount("https://", adapter)
        
        # User-Agent mimics a real browser
        _session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        })
    return _session


def get_company_name(symbol: str) -> str:
    return SYMBOL_NAMES.get(symbol, symbol)


def _api_get(endpoint: str, params: dict | None = None) -> dict:
    """
    Finnhub GET with rate-limit guard, retry logic, and exponential backoff.
    Used for /quote only.
    """
    global _request_count
    if _request_count >= MAX_REQUESTS:
        print(f"  ⚠ Finnhub budget ({MAX_REQUESTS}) exhausted — skipping.")
        return {}
    
    _request_count += 1
    params = {**(params or {}), "token": FINNHUB_API_KEY}
    session = _get_session()
    
    try:
        r = session.get(
            f"{FINNHUB_BASE_URL}{endpoint}", 
            params=params, 
            timeout=15
        )
        r.raise_for_status()
        data = r.json()
        
        # Validate response has quote data
        if data.get("c") is None or data.get("pc") is None:
            return {}
        return data
        
    except Exception as exc:
        print(f"  ✗ Finnhub error: {exc!s:.80}")
        return {}
    finally:
        time.sleep(REQUEST_DELAY)


def fetch_quote(symbol: str) -> dict:
    """Current-day quote via Finnhub /quote (free tier)."""
    raw = _api_get("/quote", {"symbol": symbol})
    return {
        "symbol":     symbol,
        "current":    raw.get("c", 0),
        "prev_close": raw.get("pc", 0),
    }


def fetch_ohlc(symbol: str, days: int = OHLC_DAYS) -> pd.DataFrame:
    """
    OHLC history via yfinance — completely free, no API key needed.
    Replaces Finnhub /stock/candle which requires a paid subscription.
    Includes retry logic with exponential backoff.
    """
    end   = datetime.now()
    start = end - timedelta(days=days + 4)   # +4 buffer for weekends/holidays
    
    max_retries = 3
    backoff_base = 0.5
    
    for attempt in range(max_retries):
        try:
            ticker = yf.Ticker(symbol)
            df = ticker.history(
                start=start.strftime("%Y-%m-%d"),
                end=end.strftime("%Y-%m-%d"),
                interval="1d",
                auto_adjust=True
            )
            
            if df.empty:
                return pd.DataFrame()

            df = df[["Open", "High", "Low", "Close"]].copy()
            df.index = pd.to_datetime(df.index).normalize().tz_localize(None)
            return df.tail(days)          # keep exactly the requested window

        except Exception as exc:
            if attempt < max_retries - 1:
                delay = backoff_base * (2 ** attempt)
                print(f"  ⚠ yfinance error ({symbol}) — retry in {delay:.1f}s", end="")
                time.sleep(delay)
                continue
            else:
                print(f"  ✗ yfinance error ({symbol}): {exc!s:.60}")
                return pd.DataFrame()
    
    return pd.DataFrame()


def _pct_change(quote: dict) -> float | None:
    """Daily % change; None if data is invalid."""
    c, pc = quote["current"], quote["prev_close"]
    if c <= 0 or pc <= 0:
        return None
    return (c - pc) / pc * 100


# Step 1 — Identify top 5 daily movers (Finnhub /quote scan)
print("=" * 65)
print(f" STEP 1 — Scanning top {SCAN_LIMIT} S&P 500 tickers for daily movers")
print(f"          Source: Finnhub /quote  (free tier)")
print(f"          Retry: Exponential backoff (3 attempts, 2s delay)")
print("=" * 65)

sp500_tickers = list(
    PyTickerSymbols().get_sp_500_nyc_yahoo_tickers()
)[:SCAN_LIMIT]

mover_scores: list[tuple[str, float]] = []

for i, sym in enumerate(sp500_tickers, 1):
    if _request_count >= MAX_REQUESTS:
        print("  ⚠ Budget exhausted during mover scan.")
        break
    print(f"  [{i:02}/{len(sp500_tickers)}] {sym:<8}", end="", flush=True)
    q   = fetch_quote(sym)
    pct = _pct_change(q)
    if pct is None:
        print(" — no data")
    else:
        mover_scores.append((sym, pct))
        print(f" {pct:+.2f}%")

print(f"\n  Valid movers found: {len(mover_scores)}")

if len(mover_scores) < 5:
    print(f"  ⚠ Found only {len(mover_scores)} movers — using defaults for missing entries")
    mover_scores.sort(key=lambda x: x[1], reverse=True)
    top5_symbols = [s for s, _ in mover_scores]
    
    defaults = ["AAPL", "MSFT", "NVDA", "TSLA", "AMZN"]
    for d in defaults:
        if len(top5_symbols) >= 5:
            break
        if d not in [s for s, _ in mover_scores]:
            top5_symbols.append(d)
else:
    mover_scores.sort(key=lambda x: x[1], reverse=True)
    top5_symbols = [s for s, _ in mover_scores[:5]]

print(f"\n  ✓ Top 5 movers : {top5_symbols}")
print(f"  Finnhub requests used: {_request_count}/{MAX_REQUESTS}\n")

# Step 2 — Fetch OHLC for top 5 movers (yfinance — no quota cost)
print("=" * 65)
print(f" STEP 2 — Fetching {OHLC_DAYS}-day OHLC for top 5 movers")
print(f"          Source: yfinance (free, no API key required)")
print(f"          Retry: Exponential backoff (3 attempts)")
print("=" * 65)

ohlc_data: dict[str, pd.DataFrame] = {}

for sym in top5_symbols:
    print(f"  {sym} ...", end="", flush=True)
    df = fetch_ohlc(sym)
    if df.empty:
        print(" ✗ no data")
    else:
        ohlc_data[sym] = df
        print(f" ✓ ({len(df)} trading days)")

print(f"\n  Successfully fetched: {len(ohlc_data)} symbols")
print(f"  Finnhub requests used: {_request_count}/{MAX_REQUESTS}\n")

# Step 3 — Watchlist: quotes (Finnhub) + OHLC (yfinance)
print("=" * 65)
print(f" STEP 3 — Fetching quotes & OHLC for watchlist")
print(f"          Quotes: Finnhub /quote  |  OHLC: yfinance")
print("=" * 65)

watchlist_performance: list[tuple[str, float]] = []
watchlist_ohlc_data:   dict[str, pd.DataFrame] = {}

for sym in WATCHLIST:
    print(f"  {sym} ...", end="", flush=True)
    q   = fetch_quote(sym)          # Finnhub (costs 1 request)
    pct = _pct_change(q)
    if pct is not None:
        watchlist_performance.append((sym, pct))

    df = fetch_ohlc(sym)            # yfinance (free, no quota)
    if not df.empty:
        watchlist_ohlc_data[sym] = df

    q_str = f"{pct:+.2f}%" if pct is not None else "no quote"
    c_str = f"{len(df)} days" if not df.empty else "no candles"
    print(f" ✓  quote={q_str}  OHLC={c_str}")

watchlist_performance.sort(key=lambda x: x[1], reverse=True)

print(f"\n  Total Finnhub requests used: {_request_count}/{MAX_REQUESTS}")
print("\n✅ All data fetched — proceed to Cells 2, 3, and 4 for charts.")

 -1.63%
  [17/50] APA     

## Summary

✅ **Data Available:**
- `top5_symbols`: List of top 5 movers
- `ohlc_data`: Dictionary with OHLC data for each top 5 symbol
- `WATCHLIST`: List of watchlist stocks (AAPL, NVDA, MSFT, META, GOOGL)
- `request_count`: Current API request count

✅ **Functions Available:**
- `get_stock_quote(symbol)`: Fetch current/previous close
- `get_stock_candles(symbol, days)`: Fetch OHLC data
- `get_company_name(symbol)`: Get company name
- All other helper functions

## Top 5 Movers - 7-Day OHLC Visualization

This section visualizes the 7-day Open-High-Low-Close (OHLC) price movements for the top 5 stocks by daily percentage change.

**Data Source**: `ohlc_data` from Cell 1
**Requirements**: Cell 1 must be executed first with successful data fetch

In [ ]:
# =============================================================================
# CELL 2 — TOP 5 MOVERS VISUALIZATIONS
# Requires: ohlc_data  (from Cell 1)
# Charts:
#   [1] Seaborn    — Top 5 Movers OHLC line charts (one per stock, faceted)
#   [2] Matplotlib — Top 5 Movers close price comparison (absolute + rebased)
# =============================================================================

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import pandas as pd

# Guard
if not globals().get("ohlc_data"):
    raise RuntimeError("ohlc_data is missing — run Cell 1 first.")

OHLC_COLORS  = {"Open": "#4C72B0", "High": "#2ecc71", "Low": "#e74c3c", "Close": "#9b59b6"}
OHLC_MARKERS = {"Open": "o",       "High": "^",        "Low": "v",       "Close": "s"}

def dollar_fmt(v, _): return f"${v:,.2f}"


# CHART 1 — Seaborn: Top 5 Movers OHLC Line Charts (faceted, one per stock)
print("Rendering Chart 1 — Top 5 Movers OHLC …")

n = len(ohlc_data)
fig, axes = plt.subplots(n, 1, figsize=(13, 4.5 * n))
axes = [axes] if n == 1 else list(axes)

sns.set_theme(style="darkgrid", palette="tab10", font_scale=1.05)

for ax, (sym, df) in zip(axes, ohlc_data.items()):
    df_long = (
        df.reset_index()
          .rename(columns={"index": "Date"})
          .melt(id_vars="Date", var_name="Price Type", value_name="Price")
    )
    for pt in ["Open", "High", "Low", "Close"]:
        sub = df_long[df_long["Price Type"] == pt]
        sns.lineplot(
            data=sub, x="Date", y="Price", ax=ax,
            label=pt, color=OHLC_COLORS[pt],
            marker=OHLC_MARKERS[pt], linewidth=2, markersize=8,
        )

    ax.fill_between(df.index, df["Low"], df["High"],
                    alpha=0.08, color="#888888", label="_nolegend_")

    company = get_company_name(sym)
    ax.set_title(f"{sym} — {company}  |  {OHLC_DAYS}-Day OHLC",
                 fontsize=13, fontweight="bold", pad=10)
    ax.set_xlabel("")
    ax.set_ylabel("Price ($)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(dollar_fmt))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.tick_params(axis="x", rotation=30)
    ax.legend(title="Price Type", loc="upper left", framealpha=0.85, ncol=4)

fig.suptitle("Top 5 Daily Movers — OHLC Line Charts  (Seaborn)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
print("  ✓ Chart 1 complete.\n")


# CHART 2 — Matplotlib: Top 5 Movers Close Price Comparison
print("Rendering Chart 2 — Top 5 Movers Close Price Comparison …")

fig, (ax_abs, ax_norm) = plt.subplots(2, 1, figsize=(13, 10))
cmap   = plt.get_cmap("tab10")
colors = {sym: cmap(i % 10) for i, sym in enumerate(ohlc_data)}

for sym, df in ohlc_data.items():
    label = f"{sym} ★"
    ax_abs.plot(df.index, df["Close"],
                color=colors[sym], linewidth=2.2,
                marker="o", markersize=5, label=label)
    rebased = df["Close"] / df["Close"].iloc[0] * 100
    ax_norm.plot(df.index, rebased,
                 color=colors[sym], linewidth=2.2,
                 marker="o", markersize=5, label=label)

ax_abs.set_title("Top 5 Movers — Close Price (Absolute $)",
                 fontsize=12, fontweight="bold")
ax_abs.set_ylabel("Price ($)")
ax_abs.yaxis.set_major_formatter(mticker.FuncFormatter(dollar_fmt))
ax_abs.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax_abs.tick_params(axis="x", rotation=30)
ax_abs.legend(fontsize=9, loc="upper left", framealpha=0.85, ncol=3)
ax_abs.grid(True, linestyle="--", alpha=0.4)

ax_norm.axhline(100, color="white", linewidth=0.9, linestyle=":")
ax_norm.set_title("Top 5 Movers — Rebased to 100 (relative performance)",
                  fontsize=12, fontweight="bold")
ax_norm.set_ylabel("Rebased Price (start = 100)")
ax_norm.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax_norm.tick_params(axis="x", rotation=30)
ax_norm.legend(fontsize=9, loc="upper left", framealpha=0.85, ncol=3)
ax_norm.grid(True, linestyle="--", alpha=0.4)

fig.suptitle(f"Top 5 Movers — Close Price Comparison  |  Last {OHLC_DAYS} Trading Days  (Matplotlib)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
print("  ✓ Chart 2 complete.\n")

print("=" * 65)
print(" ✅ Cell 2 complete — run Cell 3 for watchlist charts.")
print("=" * 65)

## Watchlist Analysis & Visualization

This section monitors your favorite stocks (AAPL, NVDA, MSFT, META, GOOGL) for:
- Daily performance (% change from previous close)
- 7-day price trends (Open, High, Low, Close)

**Execution Flow**:
- Fetch watchlist performance & OHLC data → Creates `watchlist_ohlc_data`
- Visualize the watchlist data using matplotlib

## Watchlist - 7-Day OHLC Visualization

Visualize your watchlist stocks with detailed OHLC charts showing 7-day price movements.

**Data Source**: `watchlist_ohlc_data` from Cell 4
**Requirements**: Cell 4 must be executed first with successful data fetch

In [ ]:
# =============================================================================
# CELL 3 — WATCHLIST VISUALIZATIONS
# Requires: watchlist_ohlc_data, watchlist_performance  (from Cell 1)
# Charts:
#   [1] Seaborn    — Watchlist daily % change bar chart
#   [2] Matplotlib — Watchlist candlestick-style OHLC subplots
#   [3] Matplotlib — Watchlist close price comparison (absolute + rebased)
# =============================================================================

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

# Guard
_missing = [n for n, v in [
    ("watchlist_ohlc_data",   globals().get("watchlist_ohlc_data")),
    ("watchlist_performance", globals().get("watchlist_performance")),
] if not v]
if _missing:
    raise RuntimeError(f"Missing from Cell 1: {_missing} — run Cell 1 first.")

def dollar_fmt(v, _): return f"${v:,.2f}"
def pct_fmt(v, _):    return f"{v:+.1f}%"

# CHART 1 — Seaborn: Watchlist Daily % Change Bar Chart
print("Rendering Chart 1 — Watchlist % Change …")

sns.set_theme(style="whitegrid", font_scale=1.1)

perf_df = pd.DataFrame(watchlist_performance, columns=["Symbol", "Pct"])
perf_df["Company"] = perf_df["Symbol"].map(get_company_name)
perf_df["Label"]   = perf_df["Symbol"] + "\n" + perf_df["Company"]
perf_df["Color"]   = perf_df["Pct"].apply(lambda v: "#2ecc71" if v >= 0 else "#e74c3c")
perf_df = perf_df.sort_values("Pct")

fig, ax = plt.subplots(figsize=(11, max(4, 0.9 * len(perf_df) + 2)))

bars = ax.barh(perf_df["Label"], perf_df["Pct"],
               color=perf_df["Color"].tolist(),
               edgecolor="white", linewidth=0.8, height=0.55)

for bar, val in zip(bars, perf_df["Pct"]):
    pad = 0.05
    ha  = "left" if val >= 0 else "right"
    ax.text(val + (pad if val >= 0 else -pad),
            bar.get_y() + bar.get_height() / 2,
            f"{val:+.2f}%", va="center", ha=ha,
            fontsize=10, fontweight="bold")

ax.axvline(0, color="gray", linewidth=1.2, linestyle="--")
ax.set_title("Watchlist — Daily % Change  (Seaborn)",
             fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Daily Change (%)")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))

gain_patch = mpatches.Patch(color="#2ecc71", label="Gain")
loss_patch = mpatches.Patch(color="#e74c3c", label="Loss")
ax.legend(handles=[gain_patch, loss_patch], loc="lower right", framealpha=0.8)

plt.tight_layout()
plt.show()
print("  ✓ Chart 1 complete.\n")

# CHART 2 — Matplotlib: Watchlist Candlestick-style OHLC Subplots
print("Rendering Chart 2 — Watchlist Candlestick OHLC …")

n     = len(watchlist_ohlc_data)
ncols = 2
nrows = (n + 1) // ncols
fig, axs = plt.subplots(nrows, ncols,
                         figsize=(14, 4.5 * nrows),
                         constrained_layout=True)
axs_flat = list(axs.flatten()) if n > 1 else [axs]

for ax, (sym, df) in zip(axs_flat, watchlist_ohlc_data.items()):
    dates = np.arange(len(df))
    width = 0.4

    for i, (_, row) in enumerate(df.iterrows()):
        color  = "#2ecc71" if row["Close"] >= row["Open"] else "#e74c3c"
        bottom = min(row["Open"], row["Close"])
        height = abs(row["Close"] - row["Open"]) or 0.01
        ax.bar(i, height, bottom=bottom, width=width,
               color=color, edgecolor="white", linewidth=0.6)
        ax.plot([i, i], [row["Low"], row["High"]],
                color=color, linewidth=1.2)

    # Close trend overlay
    ax.plot(dates, df["Close"].values, color="#9b59b6",
            linewidth=1.5, linestyle="--", alpha=0.7, label="Close trend")

    company = get_company_name(sym)
    ax.set_title(f"{sym} — {company}", fontsize=11,
                 fontweight="bold", color="#1a252f")
    ax.set_ylabel("Price ($)", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(dollar_fmt))
    ax.set_xticks(dates)
    ax.set_xticklabels(
        [d.strftime("%b %d") for d in df.index],
        rotation=35, ha="right", fontsize=8
    )
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(True, linestyle="--", alpha=0.35)

    up_p   = mpatches.Patch(color="#2ecc71", label="Bullish")
    down_p = mpatches.Patch(color="#e74c3c", label="Bearish")
    ax.legend(handles=[up_p, down_p,
                        plt.Line2D([0], [0], color="#9b59b6",
                                   linestyle="--", label="Close trend")],
              fontsize=7, loc="upper left", framealpha=0.8)

for ax in axs_flat[n:]:
    ax.set_visible(False)

fig.suptitle(
    f"Watchlist — Candlestick OHLC  |  Last {OHLC_DAYS} Trading Days  (Matplotlib)",
    fontsize=13, fontweight="bold"
)
plt.show()
print("  ✓ Chart 2 complete.\n")

# CHART 3 — Matplotlib: Watchlist Close Price Comparison (absolute + rebased)
print("Rendering Chart 3 — Watchlist Close Price Comparison …")

fig, (ax_abs, ax_norm) = plt.subplots(2, 1, figsize=(13, 10))
cmap   = plt.get_cmap("tab10")
colors = {sym: cmap(i % 10) for i, sym in enumerate(watchlist_ohlc_data)}

for sym, df in watchlist_ohlc_data.items():
    ax_abs.plot(df.index, df["Close"],
                color=colors[sym], linewidth=2,
                linestyle="--", marker="o", markersize=5, label=sym)
    rebased = df["Close"] / df["Close"].iloc[0] * 100
    ax_norm.plot(df.index, rebased,
                 color=colors[sym], linewidth=2,
                 linestyle="--", marker="o", markersize=5, label=sym)

ax_abs.set_title("Watchlist — Close Price (Absolute $)",
                 fontsize=12, fontweight="bold")
ax_abs.set_ylabel("Price ($)")
ax_abs.yaxis.set_major_formatter(mticker.FuncFormatter(dollar_fmt))
ax_abs.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax_abs.tick_params(axis="x", rotation=30)
ax_abs.legend(fontsize=9, loc="upper left", framealpha=0.85, ncol=3)
ax_abs.grid(True, linestyle="--", alpha=0.4)

ax_norm.axhline(100, color="gray", linewidth=0.9, linestyle=":")
ax_norm.set_title("Watchlist — Rebased to 100 (relative performance)",
                  fontsize=12, fontweight="bold")
ax_norm.set_ylabel("Rebased Price (start = 100)")
ax_norm.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax_norm.tick_params(axis="x", rotation=30)
ax_norm.legend(fontsize=9, loc="upper left", framealpha=0.85, ncol=3)
ax_norm.grid(True, linestyle="--", alpha=0.4)

fig.suptitle(f"Watchlist — Close Price Comparison  |  Last {OHLC_DAYS} Trading Days  (Matplotlib)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
print("  ✓ Chart 3 complete.\n")

print("=" * 65)
print(" ✅ Cell 3 complete — all watchlist charts rendered.")
print("=" * 65)